# XML Sales Data Warehouse
Run from this project directory after executing the SQL setup. See README.md for prerequisites and full-refresh behavior.

### 1. Data Ingestion 

In [ ]:
### 3 Files (XML) --> 3 DFs

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

In [ ]:
tree = ET.parse("./Dataset/customers.xml")
root = tree.getroot()

customer_records = []

for customer in root.findall('customer'): 
    data  = {
        'customer_id' : customer.findtext('customer_id'),
        'first_name' : customer.findtext('first_name'),
        'last_name' : customer.findtext('last_name'),
        'email' : customer.findtext('email'),
        'city' : customer.findtext('city')
    }
    customer_records.append(data) 
customers_df = pd.DataFrame(customer_records)
customers_df.head(5)    

In [ ]:
tree = ET.parse("./Dataset/products.xml")
root = tree.getroot()

product_records = []

for product in root.findall('product'): 
    data  = {
        'product_id' : product.findtext('product_id'),
        'product_name' : product.findtext('product_name'),
        'category' : product.findtext('category'),
        'brand' : product.findtext('brand'),
        'price' : product.findtext('price')
    }
    product_records.append(data) 
products_df = pd.DataFrame(product_records)
products_df.head()   

In [ ]:
tree = ET.parse("./Dataset/orders.xml")
root = tree.getroot()

order_records = []

for order in root.findall('order'): 
    data  = {
        'order_id' : order.findtext('order_id'),
        'customer_id' : order.findtext('customer_id'),
        'product_id' : order.findtext('product_id'),
        'quantity' : order.findtext('quantity'),
        'order_date' : order.findtext('order_date')
    }
    order_records.append(data) 
orders_df = pd.DataFrame(order_records)
orders_df.head()

In [ ]:
orders_df
customers_df
products_df

In [ ]:
orders_df.columns

### 2. Bronze Layer

In [ ]:
### 3 DFs --> SQL Server (3 Tables Bronze)

In [ ]:
### Pip install sqlalchemy

In [ ]:
from sqlalchemy import create_engine, text

warehouse_engine = create_engine(
    "mssql+pyodbc://localhost/project1_dwh"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:
with warehouse_engine.begin() as connection:
    connection.execute(text("DELETE FROM bronze.customers"))
    customers_df.to_sql(
        'customers', 
        con = connection, 
        schema = 'bronze',
        index = False, 
        if_exists = 'append'
        )


In [ ]:
with warehouse_engine.begin() as connection:
    connection.execute(text("DELETE FROM bronze.products"))
    products_df.to_sql(
        'products', 
        con = connection, 
        schema = 'bronze',
        index = False, 
        if_exists = 'append'
        )


In [ ]:
with warehouse_engine.begin() as connection:
    connection.execute(text("DELETE FROM bronze.orders"))
    orders_df.to_sql(
        'orders', 
        con = connection, 
        schema = 'bronze',
        index = False, 
        if_exists = 'append'
        )


### 3. Silver Layer

In [ ]:
### SQL Server (3 Tables Bronze) --> SQL Server (3 Tables Silver)

In [ ]:
from sqlalchemy import create_engine, text

warehouse_engine = create_engine(
    "mssql+pyodbc://localhost/project1_dwh"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:
customers_df = pd.read_sql( 'SELECT * FROM bronze.customers', con = warehouse_engine)
products_df = pd.read_sql( 'SELECT * FROM bronze.products', con = warehouse_engine)
orders_df = pd.read_sql( 'SELECT * FROM bronze.orders', con = warehouse_engine)

In [ ]:
customers_df.head()

In [ ]:
customers_df.dtypes

In [ ]:
customers_df['customer_id'] = customers_df['customer_id'].astype(int)

In [ ]:
customers_df.dtypes

In [ ]:
customers_df['full_name'] = customers_df['first_name'] + ' ' + customers_df['last_name']

In [ ]:
customers_df.drop(columns=['first_name', 'last_name'], inplace=True)

In [ ]:
customers_df.isnull().sum()

In [ ]:
products_df.dtypes

In [ ]:
products_df['product_id'] = products_df['product_id'].astype(int)
products_df['price'] = products_df['price'].astype(float)

In [ ]:
products_df

In [ ]:
orders_df.dtypes

In [ ]:
orders_df['order_id'] = orders_df['order_id'].astype(int)
orders_df['customer_id'] = orders_df['customer_id'].astype(int)
orders_df['product_id'] = orders_df['product_id'].astype(int)
orders_df['quantity'] = orders_df['quantity'].astype(int)
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

In [ ]:
customers_df
products_df
orders_df

In [ ]:
with warehouse_engine.begin() as connection:
    connection.execute(text("DELETE FROM silver.customers"))
    customers_df.to_sql(
        'customers', 
        con = connection, 
        schema = 'silver',
        index = False, 
        if_exists = 'append'
        )


In [ ]:
with warehouse_engine.begin() as connection:
    connection.execute(text("DELETE FROM silver.products"))
    products_df.to_sql(
        'products', 
        con = connection, 
        schema = 'silver',
        index = False, 
        if_exists = 'append'
        )


In [ ]:
with warehouse_engine.begin() as connection:
    connection.execute(text("DELETE FROM silver.orders"))
    orders_df.to_sql(
        'orders', 
        con = connection, 
        schema = 'silver',
        index = False, 
        if_exists = 'append'
        )


### 4. Gold Layer

In [ ]:
### SQL Server (3 Tables Silver) --> SQL Server (One Big Table Gold)

In [ ]:
from sqlalchemy import create_engine, text

warehouse_engine = create_engine(
    "mssql+pyodbc://localhost/project1_dwh"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:
customers_df = pd.read_sql( 'SELECT * FROM silver.customers', con = warehouse_engine)
products_df = pd.read_sql( 'SELECT * FROM silver.products', con = warehouse_engine)
orders_df = pd.read_sql( 'SELECT * FROM silver.orders', con = warehouse_engine)

In [ ]:
sales_fact_df = pd.merge(orders_df, customers_df, how = 'inner', on = 'customer_id')
sales_fact_df = pd.merge(sales_fact_df, products_df, how = 'inner', on = 'product_id')
sales_fact_df.head()

In [ ]:
sales_fact_df['total_amount'] = sales_fact_df['quantity'] * sales_fact_df['price']

In [ ]:
sales_fact_df.head()

In [ ]:
sales_fact_df.columns

In [ ]:
with warehouse_engine.begin() as connection:
    connection.execute(text("DELETE FROM gold.fact"))
    sales_fact_df.to_sql(
        'fact', 
        con = connection, 
        schema = 'gold',
        index = False, 
        if_exists = 'append'
        )
